In [ ]:

import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import pickle

try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

from io_utils import ensure_directory
from energy_env import EnergyEnvironment
from HEMS_controller import HEMSEnvironment
from td3_agent import TD3Agent
from sac_agent import SACAgent
from acktr_agent import ACKTRAgent

# Import improved baselines
from MPCBaseline_improved_v2 import MPCBaselineImprovedV2 as MPCBaselineImproved
from Rule_Based_Baseline_improved import RuleBasedBaselineImproved

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def safe_div(a, b, eps=1e-8):
    return a / (b + eps)


def find_td3_checkpoint(checkpoint_base_dir, seed):
    seed_dir = os.path.join(checkpoint_base_dir, f"seed_{seed}")
    if not os.path.exists(seed_dir):
        return None
    if os.path.exists(os.path.join(seed_dir, "actor_ep500.pth")):
        return seed_dir
    return None


def find_sac_checkpoint(checkpoint_base_dir, seed):
    seed_dir = os.path.join(checkpoint_base_dir, f"seed_{seed}")
    if not os.path.exists(seed_dir):
        return None
    if os.path.exists(os.path.join(seed_dir, "actor_ep500.pth")):
        return seed_dir
    return None


def find_acktr_checkpoint(checkpoint_base_dir):
    episode_dir = os.path.join(checkpoint_base_dir, "episode_500")
    if os.path.exists(episode_dir) and os.path.exists(os.path.join(episode_dir, "actor.pth")):
        return episode_dir
    return None


def load_sac_agent_direct(agent, checkpoint_dir):
    actor_path = os.path.join(checkpoint_dir, "actor_ep500.pth")
    critic1_path = os.path.join(checkpoint_dir, "critic1_ep500.pth")
    critic2_path = os.path.join(checkpoint_dir, "critic2_ep500.pth")
    
    if os.path.exists(actor_path):
        agent.actor.load_state_dict(torch.load(actor_path, map_location=DEVICE))
        print(f"[INFO] Loaded actor from {actor_path}")
    else:
        return False
    
    if os.path.exists(critic1_path):
        agent.critic1.load_state_dict(torch.load(critic1_path, map_location=DEVICE))
        print(f"[INFO] Loaded critic1 from {critic1_path}")
    else:
        return False
    
    if os.path.exists(critic2_path):
        agent.critic2.load_state_dict(torch.load(critic2_path, map_location=DEVICE))
        print(f"[INFO] Loaded critic2 from {critic2_path}")
    else:
        return False
    
    return True


class TestArgs:
    def __init__(self, algorithm_name):
        self.lr_actor = 1e-4
        self.lr_critic = 1e-4
        self.gamma = 0.99
        self.tau = 0.005
        self.policy_noise = 0.2
        self.noise_clip = 0.5
        self.policy_delay = 2
        self.batch_size = 128
        self.buffer_size = 500000
        self.alpha = 0.2
        self.alpha_tuning = True
        self.lr_alpha = 3e-4
        
        if algorithm_name == 'TD3':
            self.hidden = 512
        elif algorithm_name == 'ACKTR':
            self.hidden = 512
        else:
            self.hidden = 256
        
        self.num_episodes = 2000
        self.max_steps_per_episode = 96
        self.checkpoint_dir = ""
        self.log_dir = "./logs"


def create_agent(algorithm_name, env, seed, checkpoint_dir, state_dim, action_dim):
    args = TestArgs(algorithm_name)
    args.seed = seed
    args.checkpoint_dir = checkpoint_dir
    
    if algorithm_name == 'TD3':
        agent = TD3Agent(name=f"{algorithm_name}_test", env=env, seed=seed, args=args,
                         state_dim=state_dim, action_dim=action_dim)
    elif algorithm_name == 'SAC':
        agent = SACAgent(env=env, args=args)
    elif algorithm_name == 'ACKTR':
        agent = ACKTRAgent(name=f"{algorithm_name}_test", env=env, seed=seed, args=args)
    else:
        raise ValueError(f"Unknown algorithm: {algorithm_name}")
    
    return agent


def evaluate_agent_on_test(agent, env, max_steps=96, algorithm_name="", seed=1, start_episode=0):
    steps_per_episode = max_steps
    env.env_data.current_step = start_episode * steps_per_episode
    obs = env.reset()
    
    step_data = {
        'step': [],
        'action_bat': [],
        'action_grid': [],
        'action_pv': [],
        'action_load': [],
        'grid_import': [],
        'grid_export': [],
        'pv_used': [],
        'pv_available': [],
        'load_requested': [],
        'load_served': [],
        'unmet_load': [],
        'soc': [],
        'reward': [],
        'energy_cost': [],
        'comfort_penalty': []
    }
    
    cum_metrics = {
        'total_reward': 0.0,
        'total_energy_cost': 0.0,
        'total_pv_used': 0.0,
        'total_pv_available': 0.0,
        'total_load_requested': 0.0,
        'total_load_served': 0.0,
        'total_unmet_load': 0.0,
        'total_comfort_penalty': 0.0,
        'soc_changes': []
    }
    
    prev_soc = env.soc
    
    for step in range(max_steps):
        try:
            if algorithm_name == 'SAC':
                action = agent.select_action(obs, deterministic=True)
            else:
                action = agent.select_action(obs, deterministic=False)
        except AttributeError:
            try:
                action = agent.act(obs, exploration=False)
            except:
                action = agent.predict_action(obs, exploration=False)
        
        if torch.is_tensor(action):
            action = action.cpu().numpy()
        
        if len(action) == 5:
            action = action[:4]
        
        next_obs, reward, done, info = env.step(action)
        
        step_data['step'].append(step)
        step_data['action_bat'].append(action[0] if len(action) > 0 else 0)
        step_data['action_grid'].append(action[1] if len(action) > 1 else 0)
        step_data['action_pv'].append(action[2] if len(action) > 2 else 0)
        step_data['action_load'].append(action[3] if len(action) > 3 else 0)
        step_data['grid_import'].append(info.get('grid_import', 0.0))
        step_data['grid_export'].append(info.get('grid_export', 0.0))
        step_data['pv_used'].append(info.get('pv_used', 0.0))
        step_data['pv_available'].append(info.get('pv_available', 0.0))
        step_data['load_requested'].append(info.get('load_requested', 0.0))
        step_data['load_served'].append(info.get('load_served', 0.0))
        step_data['unmet_load'].append(info.get('unmet_load', 0.0))
        step_data['soc'].append(env.soc)
        step_data['reward'].append(reward)
        step_data['energy_cost'].append(info.get('energy_cost', 0.0))
        step_data['comfort_penalty'].append(info.get('comfort_penalty', 0.0))
        
        cum_metrics['total_reward'] += reward
        cum_metrics['total_energy_cost'] += info.get('energy_cost', 0.0)
        cum_metrics['total_pv_used'] += info.get('pv_used', 0.0)
        cum_metrics['total_pv_available'] += info.get('pv_available', 0.0)
        cum_metrics['total_load_requested'] += info.get('load_requested', 0.0)
        cum_metrics['total_load_served'] += info.get('load_served', 0.0)
        cum_metrics['total_unmet_load'] += info.get('unmet_load', 0.0)
        cum_metrics['total_comfort_penalty'] += info.get('comfort_penalty', 0.0)
        cum_metrics['soc_changes'].append(abs(env.soc - prev_soc))
        
        prev_soc = env.soc
        obs = next_obs
        
        if done:
            break
    
    total_demand = cum_metrics['total_load_requested']
    total_grid_import = sum(step_data['grid_import'])
    
    performance_metrics = {
        'seed': seed,
        'algorithm': algorithm_name,
        'total_reward': cum_metrics['total_reward'],
        'total_energy_cost': cum_metrics['total_energy_cost'],
        'SCR': safe_div(cum_metrics['total_pv_used'], cum_metrics['total_pv_available']),
        'SSR': safe_div(total_demand - total_grid_import, total_demand),
        'PAR': safe_div(max(step_data['grid_import']), np.mean(step_data['grid_import']) if np.mean(step_data['grid_import']) > 0 else 1.0),
        'battery_cycles': 0.5 * np.sum(cum_metrics['soc_changes']),
        'ULR': safe_div(cum_metrics['total_unmet_load'], total_demand),
        'CVI': cum_metrics['total_comfort_penalty'],
    }
    
    return step_data, performance_metrics


def run_baseline_comparison(env, n_episodes=10):
    """Run improved baselines for comparison"""
    print("\n[Baseline] Running improved MPC...")
    mpc_agent = MPCBaselineImproved(horizon=12)
    mpc_results = []
    
    for ep in range(n_episodes):
        env.env_data.current_step = ep * 96
        obs = env.reset()
        episode_grid = []
        
        prev_P_ch = 0
        prev_P_dis = 0
        prev_Q_HVAC = 0
        
        for step in range(96):
            if isinstance(obs, dict):
                current_SoC = obs.get('soc', 0.5) * 10.0
                P_load = obs.get('load_requested', 2.0)
                P_pv = obs.get('pv_available', 1.0)
                T_out = obs.get('T_out', 20.0)
                Q_occup = 0.2
            else:
                current_SoC = 5.0
                P_load = 2.0
                P_pv = 1.0
                T_out = 20.0
                Q_occup = 0.2
            
            H = 12  # Match MPC horizon
            P_load_forecast = [P_load] * H
            P_pv_forecast = [P_pv] * H
            T_out_forecast = [T_out] * H
            Q_occup_forecast = [Q_occup] * H
            
            # ============================================================
            # IMPROVED: Use actual price data from environment for forecasts
            # ============================================================
            price_forecast = []
            hour = step % 24
            
            for i in range(H):
                future_step = step + i
                future_obs = env.env_data.get_current_observation(future_step)
                if future_obs is not None:
                    f = env.feature_map
                    future_price = future_obs[f["price"]]
                    # Convert from $/MWh to $/kWh if needed
                    if future_price > 1.0:
                        future_price = future_price / 1000
                    price_forecast.append(future_price)
                else:
                    # Fallback to TOU schedule
                    h = (hour + i) % 24
                    if 14 <= h <= 19:
                        price_forecast.append(0.25)
                    elif 9 <= h < 14 or 19 <= h < 22:
                        price_forecast.append(0.15)
                    else:
                        price_forecast.append(0.10)
            # ============================================================
            
            # Call solve with all 9 arguments
            bat_action, _, _ = mpc_agent.solve(
                current_SoC, P_load_forecast, P_pv_forecast,
                T_out_forecast, Q_occup_forecast, price_forecast,
                prev_P_ch, prev_P_dis, prev_Q_HVAC
            )
            
            action = np.zeros(4)
            action[0] = bat_action
            action[2] = min(P_pv, P_load + max(0, bat_action))
            
            # Update previous actions
            if bat_action > 0:
                prev_P_ch = bat_action
                prev_P_dis = 0
            else:
                prev_P_ch = 0
                prev_P_dis = -bat_action if bat_action < 0 else 0
            prev_Q_HVAC = 0
            
            obs, reward, done, info = env.step(action)
            episode_grid.append(info.get('grid_import', 0))
            if done:
                break
        
        mpc_results.append(episode_grid)
        print(f"  MPC Episode {ep+1}: Avg Grid = {np.mean(episode_grid):.2f} kW")
    
    print("\n[Baseline] Running improved Rule-Based...")
    rule_agent = RuleBasedBaselineImproved()
    rule_results = []
    
    for ep in range(n_episodes):
        env.env_data.current_step = ep * 96
        obs = env.reset()
        episode_grid = []
        
        for step in range(96):
            if isinstance(obs, dict):
                SoC = obs.get('soc', 0.5)
                P_load = obs.get('load_requested', 2.0)
                P_pv = obs.get('pv_available', 1.0)
                T_in = obs.get('T_in', 22.0)
                
                current_obs = env.env_data.get_current_observation(step)
                if current_obs is not None:
                    f = env.feature_map
                    current_price = current_obs[f["price"]]
                    if current_price > 1.0:
                        current_price = current_price / 1000
                else:
                    current_price = 0.15
            else:
                SoC = 0.5
                P_load = 2.0
                P_pv = 1.0
                T_in = 22.0
                current_price = 0.15
            
            bat_action, hvac_action, pv_used = rule_agent.get_action(SoC, P_load, P_pv, T_in, 1, current_price)
            action = np.zeros(4)
            action[0] = bat_action
            action[2] = pv_used
            
            obs, reward, done, info = env.step(action)
            episode_grid.append(info.get('grid_import', 0))
            if done:
                break
        
        rule_results.append(episode_grid)
        print(f"  Rule-Based Episode {ep+1}: Avg Grid = {np.mean(episode_grid):.2f} kW")
    
    print("\n[Baseline] Running No HEMS...")
    no_hems_results = []
    
    for ep in range(n_episodes):
        env.env_data.current_step = ep * 96
        obs = env.reset()
        episode_grid = []
        
        for step in range(96):
            action = np.zeros(4)
            obs, reward, done, info = env.step(action)
            episode_grid.append(info.get('grid_import', 0))
            if done:
                break
        
        no_hems_results.append(episode_grid)
        print(f"  No HEMS Episode {ep+1}: Avg Grid = {np.mean(episode_grid):.2f} kW")
    
    return {
        'MPC': mpc_results,
        'RuleBased': rule_results,
        'NoHEMS': no_hems_results
    }


def main():
    SEEDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    ALGORITHMS = ['TD3', 'SAC', 'ACKTR']
    
    START_EPISODE = 0
    N_EVAL_EPISODES = 10
    
    base_dir = r"C:\Users\a\Desktop\my revised paper\Continuous\Impelimention"
    
    checkpoint_dirs = {
        'TD3': os.path.join(base_dir, 'checkpoints_TD3'),
        'SAC': os.path.join(base_dir, 'checkpoints_SAC'),
        'ACKTR': os.path.join(base_dir, 'checkpoints_ACKTR')
    }
    
    output_dir = './test_evaluation_results_10seeds'
    ensure_directory(output_dir)
    
    print("="*80)
    print("HEMS TEST EVALUATION - IMPROVED BASELINES")
    print(f"Starting from episode: {START_EPISODE}")
    print(f"Evaluating {N_EVAL_EPISODES} episodes per seed")
    print("="*80)
    
    data_cfg = {
        "data_dir": r"C:\Users\a\Desktop\my revised paper\Continuous\dataset",
        "filename": "hems_2024_clean_hourly.csv",
        "start_date": "2024-01-01 00:00:00",
        "end_date": "2024-12-31 23:00:00",
        "window_length": 24,
        "freq": "H",
        "split": "test",
        "train_ratio": 0.8,
    }
    
    env_cfg = {
        "window_length": 24,
        "delta_t": 1.0,
        "battery_capacity": 10.0,
        "p_bat_max": 6.0,
        "soc_min": 0.05,
        "soc_max": 0.95,
        "p_grid_import_max": 10.0,
        "p_grid_export_max": 3.0,
        "lambda_T": 0.05,
        "lambda_deg": 0.002,
        "lambda_uns": 10.0,
        "lambda_sw": 0.001,
        "lambda_pv": 0.05,
        "T_set": 22.0,
        "dead_band": 0.5,
    }
    
    print("\n[1/5] Creating test environment...")
    energy_env = EnergyEnvironment(**data_cfg)
    env = HEMSEnvironment(env_cfg, energy_env)
    
    obs_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    print(f"[INFO] Obs dim: {obs_dim}, Action dim: {action_dim}")
    
    print("\n[2/5] Running improved baselines...")
    baseline_results = run_baseline_comparison(env, n_episodes=N_EVAL_EPISODES)
    
    with open(os.path.join(output_dir, 'baseline_results_improved.pkl'), 'wb') as f:
        pickle.dump(baseline_results, f)
    print("[SAVED] Baseline results")
    
    all_metrics = {algo: [] for algo in ALGORITHMS}
    all_step_data = {algo: [] for algo in ALGORITHMS}
    
    print("\n[3/5] Evaluating DRL algorithms across 10 seeds...")
    
    for algo in ALGORITHMS:
        print(f"\n{'='*50}")
        print(f"Evaluating {algo}")
        print(f"{'='*50}")
        
        checkpoint_base = checkpoint_dirs[algo]
        
        if not os.path.exists(checkpoint_base):
            print(f"[WARNING] Checkpoint directory not found: {checkpoint_base}")
            continue
        
        successful_seeds = 0
        
        for seed in SEEDS:
            if algo == 'TD3':
                checkpoint_dir = find_td3_checkpoint(checkpoint_base, seed)
            elif algo == 'SAC':
                checkpoint_dir = find_sac_checkpoint(checkpoint_base, seed)
            else:
                checkpoint_dir = find_acktr_checkpoint(checkpoint_base)
            
            if checkpoint_dir is None:
                print(f"[WARNING] No checkpoint found for {algo} seed {seed}")
                continue
            
            try:
                agent = create_agent(algo, env, seed, checkpoint_dir, obs_dim, action_dim)
                
                seed_step_data = []
                seed_metrics = []
                
                for ep_idx in range(N_EVAL_EPISODES):
                    current_episode = START_EPISODE + ep_idx
                    
                    if algo == 'SAC':
                        if load_sac_agent_direct(agent, checkpoint_dir):
                            step_data, metrics = evaluate_agent_on_test(
                                agent, env, max_steps=96, 
                                algorithm_name=algo, seed=seed,
                                start_episode=current_episode
                            )
                            seed_step_data.append(step_data)
                            seed_metrics.append(metrics)
                        else:
                            print(f"  Seed {seed}, Ep {current_episode}: Failed to load model")
                            break
                    else:
                        try:
                            agent.load_models(checkpoint_dir)
                            step_data, metrics = evaluate_agent_on_test(
                                agent, env, max_steps=96, 
                                algorithm_name=algo, seed=seed,
                                start_episode=current_episode
                            )
                            seed_step_data.append(step_data)
                            seed_metrics.append(metrics)
                        except Exception as e:
                            print(f"  Seed {seed}, Ep {current_episode}: Failed to load - {e}")
                            break
                    
                    print(f"  Seed {seed}, Ep {current_episode}: SCR={metrics['SCR']:.4f}, SSR={metrics['SSR']:.4f}, Cost=${metrics['total_energy_cost']:.2f}")
                
                if seed_metrics:
                    all_metrics[algo].extend(seed_metrics)
                    all_step_data[algo].extend(seed_step_data)
                    successful_seeds += 1
                    avg_cost = np.mean([m['total_energy_cost'] for m in seed_metrics])
                    print(f"  Seed {seed} completed: {len(seed_metrics)} episodes, Avg Cost=${avg_cost:.2f}")
                    
            except Exception as e:
                print(f"  Seed {seed}: ERROR - {e}")
                continue
        
        if all_metrics[algo]:
            costs = [m['total_energy_cost'] for m in all_metrics[algo]]
            print(f"\n{algo} Test Results: ${np.mean(costs):.2f} ± ${np.std(costs):.2f}")
    
    print("\n[4/5] Saving results...")
    
    for algo in ALGORITHMS:
        if all_step_data[algo]:
            step_data_to_save = []
            for seed_data in all_step_data[algo]:
                serializable = {}
                for key, value in seed_data.items():
                    if isinstance(value, list):
                        serializable[key] = value
                    elif isinstance(value, np.ndarray):
                        serializable[key] = value.tolist()
                    else:
                        serializable[key] = value
                step_data_to_save.append(serializable)
            
            with open(os.path.join(output_dir, f'{algo}_step_data.pkl'), 'wb') as f:
                pickle.dump(step_data_to_save, f)
            print(f"[SAVED] {algo}_step_data.pkl")
    
    # Save baseline results summary
    baseline_summary = {}
    for name, data in baseline_results.items():
        avg_grid = np.mean([np.mean(ep) for ep in data])
        baseline_summary[name] = {'avg_grid': avg_grid}
    
    with open(os.path.join(output_dir, 'baseline_summary.pkl'), 'wb') as f:
        pickle.dump(baseline_summary, f)
    
    print("\n" + "="*80)
    print("TEST EVALUATION COMPLETE")
    print("="*80)
    
    # Print final summary
    print("\n" + "="*60)
    print("FINAL SUMMARY")
    print("="*60)
    for name, data in baseline_results.items():
        avg_grid = np.mean([np.mean(ep) for ep in data])
        print(f"{name}: Avg Grid = {avg_grid:.3f} kW")
    
    for algo in ALGORITHMS:
        if all_metrics[algo]:
            costs = [m['total_energy_cost'] for m in all_metrics[algo]]
            print(f"{algo}: Avg Cost = ${np.mean(costs):.2f} ± ${np.std(costs):.2f}")


if __name__ == "__main__":
    main()